# Cálculo de servicios de ajuste

## 📌 Descripción general

Este programa calcula los **servicios de ajuste** asociados a la factura de indexados Passthrough, a partir de los ficheros de **Liquidación Común (C2)**.

El cálculo se realiza a nivel **cuartohorario**, obteniendo tanto los **componentes individuales** como el valor final de **PRDEMCAD**, que posteriormente se exporta a Excel.

---
## ⚙️ Funcionamiento

El programa procesa los ficheros de liquidación de REE (formato C2) para calcular el **precio de demanda por servicios de ajuste (PRDEMCAD)** y sus componentes.

1. **Carga de datos**  
   El script localiza el archivo `.zip` descargado de REE, elimina ejecuciones anteriores y descomprime automáticamente todos los ficheros de trabajo.

2. **Clasificación de variables**  
   A partir del nombre de cada fichero, se identifica la variable correspondiente y se clasifica como:
   - Horaria (día × hora)
   - Cuarto-horaria (día × hora × cuarto), usada directamente o agregada a horario según el concepto.

3. **Construcción del calendario**  
   Se detectan automáticamente:
   - El número real de días del mes
   - La fecha inicial  
   Con ello se genera la estructura completa de fechas, horas y cuartos de hora.

4. **Cálculo de conceptos**  
   Se calculan individualmente todos los conceptos de servicios de ajuste (restricciones, balance, regulación, desvíos, intercambios, etc.) aplicando las fórmulas oficiales de liquidación.

5. **Cálculo del PRDEMCAD**  
   El PRDEMCAD se obtiene como la suma de todos los conceptos anteriores, calculados a nivel horario y distribuidos a cuarto de hora.

6. **Control de coherencia**  
   Se calcula la diferencia entre el PRDEMCAD y la suma de los conceptos para cada cuarto de hora.  
   Si la diferencia total no es cero, el programa emite una advertencia.

7. **Exportación**  
   Se genera un fichero Excel con:
   - Fecha, hora y cuarto de hora
   - Todos los conceptos de ajuste
   - PRDEMCAD
   - Diferencia de control
   - Variables cuarto-horarias originales (desvíos, códigos y KEST)

El resultado permite **analizar, validar y auditar** la liquidación de servicios de ajuste de forma sencilla y transparente.

---

## 🧮 Componentes calculados individualmente

El script calcula e imprime los siguientes componentes de coste/ingreso:

- **RT3**: Coste de restricciones PDBF  
- **RT6**: Coste de restricciones en tiempo real  
- **CT3**: Coste por control de tensión  
- **CFP**: Coste por factor de potencia *(ingreso para la demanda)*  
- **BALX**: Incumplimiento de balance *(ingreso para la demanda)*  
- **SECX**: Incumplimiento de regulación secundaria *(ingreso para la demanda)*  
- **BS3**: Coste de reserva de regulación  *(antigua banda de regulación secundaria)*  
- **RAD3**: Coste del servicio SRAD  
- **EXD**: Excedente / déficit de desvíos  
- **IN3**: Intercambios de apoyo  
- **IN7**: Intercambios internacionales *(PO 14.6)*  

---

## 📤 Resultados

El programa:

- Calcula todos los componentes anteriores a nivel **cuartohorario**
- Imprime los **desvíos y pérdidas**
- Genera un fichero Excel con los resultados finales listo para su uso en la facturación



In [8]:
import os
import glob
import shutil
import zipfile
import pandas as pd
import numpy as np

# =========================================================
# 1️⃣ CONFIGURACIÓN GENERAL
# =========================================================

RUTA = r"C:\Archivos origen" 
PATRON = os.path.join(RUTA, "*_liquicomun*.zip")
EXTRACT_DIR = "liq_data"

# ---------------------------------------------------------
# Buscar ZIP de entrada
# ---------------------------------------------------------
zips = glob.glob(PATRON)
if not zips:
    raise FileNotFoundError(f"No se encontró ningún ZIP en {PATRON}")

ZIP_PATH = zips[0]

# ---------------------------------------------------------
# Limpiar y extraer ZIP
# ---------------------------------------------------------
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("✅ ZIP extraído correctamente")

# =========================================================
# 2️⃣ DEFINICIÓN DE VARIABLES
# =========================================================

# Variables horarias (D × 24)
horario_vars = [
    "enrepscf", "imrad","imeradx", "imscrpbf", "imscrtre",
    "imctd", "impdcfp", "imincbal", "imexdedv",
    "scappre", "imeddire", "prdemcad", "petar6p"]

# Variables QH que se usan TAL CUAL (D × 24 × 4)
cuartoh_vars_raw = [
    "kestimqh",
    "codsvbaqh",
    "codsvsuqh",
    "imresecqh",
    "endvbrpqh",
    "enrepscqh",
    "pmdiario"
]

# Variables QH que se agregan a horario (D × 24)
cuartoh_vars_sum = ["imsecxqh"]

# =========================================================
# 3️⃣ FUNCIONES DE LECTURA DE FICHEROS
# =========================================================

def leer_c2_horario(path, dias_totales):
    """
    Lee un fichero horario C2 y devuelve array (dias, 24)
    """
    df = pd.read_csv(
        path, sep=";", decimal=",", skiprows=2,
        header=None, encoding="latin1",
        engine="python", on_bad_lines="skip"
    )

    # columnas 1..24 = horas
    df = df.iloc[:, 1:25].fillna(0).astype(float)

    # eliminar posible fila final '*'
    df = df.iloc[:dias_totales]

    return df.values


def leer_c2_cuartoh_raw(path, fechas_ref):
    """
    Lee un fichero cuartohorario C2 y devuelve array (dias, 24, 4)
    """
    df = pd.read_csv(
        path, sep=";", decimal=",", skiprows=2,
        header=None, encoding="latin1",
        engine="python", on_bad_lines="skip"
    )

    df = df.iloc[:, :4]
    df.columns = ["fecha", "hora", "cuarto", "valor"]

    # eliminar fila final '*'
    df = df[df["fecha"] != "*"]

    # tipado seguro
    df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors="coerce")
    df["hora"] = pd.to_numeric(df["hora"], errors="coerce")
    df["cuarto"] = pd.to_numeric(df["cuarto"], errors="coerce")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce").fillna(0)

    df = df.dropna(subset=["fecha", "hora", "cuarto"])

    arr = np.zeros((len(fechas_ref), 24, 4), dtype=float)
    fecha_idx = {f: i for i, f in enumerate(fechas_ref)}

    for _, r in df.iterrows():
        f = r["fecha"].normalize()
        if f in fecha_idx and 1 <= r["hora"] <= 24 and 1 <= r["cuarto"] <= 4:
            d = fecha_idx[f]
            h = int(r["hora"]) - 1
            q = int(r["cuarto"]) - 1
            arr[d, h, q] += r["valor"]

    return arr


def leer_c2_cuartoh_sum(path, fechas_ref):
    """
    Lee cuartohorario y agrega a horario (suma de cuartos)
    """
    return leer_c2_cuartoh_raw(path, fechas_ref).sum(axis=2)

# =========================================================
# 4️⃣ CARGA Y DETECCIÓN DE ESTRUCTURA TEMPORAL
# =========================================================

files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*_*"), recursive=True)
if not files:
    raise Exception("No se encontraron ficheros extraídos")

data_horario = {}
data_cuartoh = {}

# ---------------------------------------------------------
# Detectar número real de días (ignorando '*')
# ---------------------------------------------------------
# ---------------------------------------------------------
# Detección del número real de días del mes
# ---------------------------------------------------------

dias_totales = None  # Inicializa el número de días reales

for f in files:  # Recorre todos los ficheros extraídos
    name = os.path.basename(f).lower()  # Nombre del fichero en minúsculas

    if any(v in name for v in horario_vars):  # Busca un fichero horario cualquiera
        df_tmp = pd.read_csv(
            f, sep=";", decimal=",", skiprows=2,
            header=None, encoding="latin1",
            engine="python", on_bad_lines="skip"
        )

        df_tmp = df_tmp[df_tmp.iloc[:, 0] != "*"]  # Elimina la fila final '*' (si existe)
        dias_totales = len(df_tmp)                # Número real de días del mes
        break                                     # Con uno basta (todos tienen mismo calendario)

if dias_totales is None:
    raise Exception("No se pudieron detectar los días reales")

print(f"📅 Días detectados: {dias_totales}")


# ---------------------------------------------------------
# Detección de la fecha inicial del mes
# ---------------------------------------------------------

fecha_ini = None  # Fecha de inicio del periodo

for f in files:  # Recorre los ficheros para extraer la fecha del nombre
    partes = os.path.basename(f).split("_")  # Divide el nombre por '_'

    # Se asume formato ..._YYYYMMDD_YYYYMMDD.ext
    if len(partes) >= 3 and partes[-2].isdigit():
        fecha_ini = pd.to_datetime(partes[-2], format="%Y%m%d")  # Fecha inicial
        break

if fecha_ini is None:
    raise Exception("No se pudo detectar la fecha inicial")


# ---------------------------------------------------------
# Construcción del calendario de referencia
# ---------------------------------------------------------

fechas_ref = pd.date_range(
    start=fecha_ini,      # Fecha inicial detectada
    periods=dias_totales, # Número real de días
    freq="D"              # Frecuencia diaria
).normalize()             # Elimina horas (00:00)


# =========================================================
# 5️⃣ CARGA DE DATOS
# =========================================================

#para buscar la variable esta aparece entre barras bajas _{v}_

for f in files: #Este bucle recorre todos los ficheros C2 encontrados en el ZIP 
    nombre = os.path.basename(f).lower() #Se queda solo con el nombre del fichero (sin ruta) y se pasa a minúsculas

    if any(f"_{v}_" in nombre for v in horario_vars): #busqueda de variables horarias (D x 24)
        v = next(v for v in horario_vars if f"_{v}_" in nombre) #Identificación exacta de la variable
        data_horario[v] = leer_c2_horario(f, dias_totales) #devuelve array (días, 24)

    elif any(f"_{v}_" in nombre for v in cuartoh_vars_raw): #busqueda de variables cuarto-horarias (D x 24 x4)
        v = next(v for v in cuartoh_vars_raw if f"_{v}_" in nombre) #Identificación exacta de la variable
        data_cuartoh[v] = leer_c2_cuartoh_raw(f, fechas_ref) #devuelve array (días, 24, 4)

    elif any(f"_{v}_" in nombre for v in cuartoh_vars_sum): #busqueda de variables cuartohorarias agregadas (D x 24)
        v = next(v for v in cuartoh_vars_sum if f"_{v}_" in nombre) #Identificación exacta de la variable
        data_cuartoh[v] = leer_c2_cuartoh_sum(f, fechas_ref) #devuelve array (días, 24)

print("✔ Horarios:", data_horario.keys())
print("✔ Cuartohorarios:", data_cuartoh.keys())

# =========================================================
# 6️⃣ ACCESORES DE ARRAYS
# =========================================================

def H(v):
    return data_horario.get(v, np.zeros((dias_totales, 24)))

def Q(v):
    return data_cuartoh.get(v, np.zeros((dias_totales, 24, 4)))

# =========================================================
# 7️⃣ ASIGNACIÓN DE VARIABLES
# =========================================================

# Construcción de índice cuarto-horario real
datetime_index = pd.date_range(
    start=fecha_ini,
    periods=dias_totales * 24 * 4,
    freq="15min"
)

fechas_col = datetime_index
horas_col = datetime_index.hour
cuarto_col = (datetime_index.minute // 15) + 1



imscrpbf  = H("imscrpbf")
imscrtre  = H("imscrtre")
imctd     = H("imctd")
impdcfp   = H("impdcfp")
imincbal  = H("imincbal")
imexdedv  = H("imexdedv")
scappre   = H("scappre")
imeddire  = H("imeddire")
enrepscf  = H("enrepscf")


imrad     = H("imrad")
imeradx   = H("imeradx")
prdemcad  = H("prdemcad")
petar6p   = H("petar6p")

imresecqh = Q("imresecqh")
endvBRPqh = Q("endvbrpqh")
enrepscqh = Q("enrepscqh")
imsecxqh  = Q("imsecxqh")

omie      = Q("pmdiario")
kestimqh  = Q("kestimqh")
codsvsuqh = Q("codsvsuqh")
codsvbaqh = Q("codsvbaqh")


# =========================================================
# 8️⃣ CÁLCULO DE SUMANDOS
# =========================================================

def div_safe(n, d):
    return np.divide(n, d, out=np.zeros_like(n, dtype=float), where=d != 0)

RT3  = div_safe(-imscrpbf, enrepscf)
RT6  = div_safe(-imscrtre, enrepscf)
CT3  = div_safe(-imctd, enrepscf)
CFP  = div_safe(-impdcfp, enrepscf)
BALX = div_safe(-imincbal, enrepscf)
SECX = div_safe(+imsecxqh, enrepscf)

# BS3
den_qh = endvBRPqh + enrepscqh
term_qh = np.divide(imresecqh * enrepscqh, den_qh,
                    out=np.zeros_like(imresecqh), where=den_qh != 0)
BS3 = div_safe(term_qh.sum(axis=2), enrepscf)

# RAD3
num_rad = imrad + imeradx
RAD3 = np.divide(
    num_rad,
    endvBRPqh.sum(axis=2) + enrepscf,
    out=np.zeros_like(imrad),
    where=(endvBRPqh.sum(axis=2) + enrepscf) != 0
)

EXD = div_safe(-imexdedv, enrepscf)
IN3 = div_safe(-scappre, enrepscf)
IN7 = div_safe(-imeddire, enrepscf)
PERIODIFICACION = petar6p
IMRAD = imrad

# =========================================================
# 9️⃣ EXPANSIÓN A CUARTO-HORARIO (D × 24 → D × 24 × 4)
# =========================================================

def expandir_qh(h):
    """
    Expande un array horario (días, 24) a cuarto-horario (días, 24, 4)
    replicando el valor horario en los 4 cuartos.
    """
    return np.repeat(h[:, :, None], 4, axis=2)


RT3_qh  = expandir_qh(RT3)
RT6_qh  = expandir_qh(RT6)
CT3_qh  = expandir_qh(CT3)
CFP_qh  = expandir_qh(CFP)
BALX_qh = expandir_qh(BALX)
SECX_qh = expandir_qh(SECX)
BS3_qh  = expandir_qh(BS3)
RAD3_qh = expandir_qh(RAD3)
EXD_qh  = expandir_qh(EXD)
IN3_qh  = expandir_qh(IN3)
IN7_qh  = expandir_qh(IN7)
PRD_qh  = expandir_qh(prdemcad)
PER_qh  = expandir_qh(PERIODIFICACION)
IMRAD_qh= expandir_qh(IMRAD)/4


# =========================================================
# 🔎 CONTROL DE COHERENCIA
# Suma de conceptos vs PRDEMCAD
# =========================================================

SUM_CONCEPTOS_qh = (
    RT3_qh +
    RT6_qh +
    CT3_qh +
    CFP_qh +
    BALX_qh +
    SECX_qh +
    BS3_qh +
    RAD3_qh +
    EXD_qh +
    IN3_qh +
    IN7_qh
)

# Redondeo SOLO para control
SUM_CONCEPTOS_ctrl = np.round(SUM_CONCEPTOS_qh, 2)
PRD_ctrl = np.round(PRD_qh, 2)

DIF_qh = SUM_CONCEPTOS_ctrl - PRD_ctrl
DIF_flat = DIF_qh.flatten()



# =========================================================
# 📤 EXPORTACIÓN A EXCEL
# =========================================================

df_export = pd.DataFrame({
    "FECHA": fechas_col,
    "HORA": horas_col,
    "CUARTO": cuarto_col,

    "Precio OMIE": omie.flatten(),

    "RT3": RT3_qh.flatten(),
    "RT6": RT6_qh.flatten(),
    "CT3": CT3_qh.flatten(),
    "CFP": CFP_qh.flatten(),
    "BALX": BALX_qh.flatten(),
    "SECX": SECX_qh.flatten(),
    "BS3": BS3_qh.flatten(),
    "RAD3": RAD3_qh.flatten(),
    "EXD": EXD_qh.flatten(),
    "IN3": IN3_qh.flatten(),
    "IN7": IN7_qh.flatten(),

    "PRDEMCAD": PRD_qh.flatten(),
    "DIF_PRDEMCAD": DIF_flat,

    # Variables QH originales (sin tocar)
    "IMRAD": IMRAD_qh.flatten(),
    "ENREP": enrepscqh.flatten(),
    "ENDBRP": endvBRPqh.flatten(),
    "CODBAJ": codsvbaqh.flatten(),
    "CODSUB": codsvsuqh.flatten(),
    "KEST":   kestimqh.flatten(),
    "Periodos": PER_qh.flatten()
})

output = os.path.join(RUTA, "prdemcad.xlsx")
df_export.to_excel(output, index=False)

print(f"✅ Excel generado correctamente en: {output}")


# =========================================================
# ⚠️ AVISO FINAL DE CONTROL
# =========================================================

dif_total = DIF_flat.sum()

if not np.isclose(dif_total, 0.0, atol=0.1):
    print("⚠️ ADVERTENCIA: diferencia total distinta de cero")
    print(f"   Σ(DIF_PRDEMCAD) = {dif_total:.6f}")
else:
    print("✅ Control OK: la suma de conceptos coincide con PRDEMCAD")



✅ ZIP extraído correctamente
📅 Días detectados: 28
✔ Horarios: dict_keys(['enrepscf', 'imeddire', 'imexdedv', 'imincbal', 'imrad', 'imscrpbf', 'imscrtre', 'petar6p', 'prdemcad'])
✔ Cuartohorarios: dict_keys(['codsvbaqh', 'codsvsuqh', 'endvbrpqh', 'enrepscqh', 'imresecqh', 'imsecxqh', 'kestimqh', 'pmdiario'])
✅ Excel generado correctamente en: C:\Archivos origen\prdemcad.xlsx
⚠️ ADVERTENCIA: diferencia total distinta de cero
   Σ(DIF_PRDEMCAD) = 16.560000
